# Notebook 05 — Arquitectura PHN + GraphSAGE

## Componentes

| Módulo | Inspiración | Rol |
|---|---|---|
| `HyperNetwork` | Navon et al. 2021, patrón `LeNetHyper` | `r → pesos del GNN` |
| `MeanAggregator` | PyG `MessagePassing` | agregación media de vecinos |
| `PHNGraphSAGE` | Shafi et al. 2025, clase `SAGE` (utils.py L663) | GNN con pesos externos |
| `PHNModel` | combinación | entrada batch → probabilidad por nodo |

### Flujo completo
```
r = (r_cov, r_cost) ──► HyperNetwork (MLP 2→100→100) ──► dict de pesos W
                                                              │
grafo (x, edge_index) ──────────────────────────────────────►│
                                                              ▼
                              SAGEConv1(x, W₁) → BN → ReLU → Dropout
                              SAGEConv2(h, W₂) → BN → ReLU → Dropout
                              Linear(h, W_out)  → Sigmoid
                                                              │
                                                    p̂ ∈ [0,1]^N por nodo
```

### Convención SAGEConv
$$h_i = \underbrace{W_{l}^\top \cdot \text{mean}_{j\in\mathcal{N}(i)}(x_j)}_{\text{lin\_l}} + \underbrace{W_{r}^\top \cdot x_i}_{\text{lin\_r}}$$

`lin_l` tiene bias; `lin_r` no tiene bias (igual que PyG SAGEConv con `root_weight=True`).

In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.loader import DataLoader

ROOT = Path("..")
sys.path.insert(0, str(ROOT / "pareto-hypernetworks"))

# Referencia PHN: pareto-hypernetworks/phn/solvers.py
# Referencia GNN: graph-scp/Graph-SCP/utils.py (clase SAGE, líneas 663-707)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")
import torch_geometric; print(f"PyG: {torch_geometric.__version__}")

device: cpu
PyG: 2.8.0


## 1. MeanAggregator

Implementa la parte de agregación de SAGEConv usando `MessagePassing` de PyG.
Aplica `mean` sobre los features de los vecinos de cada nodo.

In [2]:
class MeanAggregator(MessagePassing):
    """Agrega features de vecinos con media (parte de SAGEConv sin pesos)."""

    def __init__(self):
        super().__init__(aggr='mean')

    def forward(self, x, edge_index, num_nodes):
        # size=(N, N) garantiza que nodos aislados (sin aristas) devuelvan 0
        return self.propagate(edge_index, x=x, size=(num_nodes, num_nodes))

    def message(self, x_j):
        return x_j   # mensajes = features del nodo fuente

# Verificación rápida
agg_test = MeanAggregator()
x_t = torch.randn(5, 3)
ei_t = torch.tensor([[0, 1, 2, 3], [1, 0, 3, 2]])  # 2 pares de nodos conectados
out_t = agg_test(x_t, ei_t, num_nodes=5)
print(f"MeanAggregator OK — input {tuple(x_t.shape)} → output {tuple(out_t.shape)}")
print(f"Nodo aislado (idx 4) debería ser 0: {out_t[4].tolist()}")

MeanAggregator OK — input (5, 3) → output (5, 3)
Nodo aislado (idx 4) debería ser 0: [0.0, 0.0, 0.0]


## 2. HyperNetwork

Mapea el vector de preferencia `r ∈ ℝ²` a los pesos de todas las capas del GNN.
Patrón: `ray_mlp` compartido + un head lineal por tensor de pesos.

Pesos generados:
| Tensor | Shape | Parámetros HPN |
|---|---|---|
| `conv1.lin_l` weight | `(h, c)` | Linear(100, h·c) |
| `conv1.lin_l` bias | `(h,)` | Linear(100, h) |
| `conv1.lin_r` weight | `(h, c)` | Linear(100, h·c) |
| `conv2.lin_l` weight | `(h, h)` | Linear(100, h·h) |
| `conv2.lin_l` bias | `(h,)` | Linear(100, h) |
| `conv2.lin_r` weight | `(h, h)` | Linear(100, h·h) |
| `final` weight | `(1, h)` | Linear(100, h) |
| `final` bias | `(1,)` | Linear(100, 1) |

In [3]:
class HyperNetwork(nn.Module):
    """
    r = (r_cov, r_cost)  →  dict de pesos para PHNGraphSAGE.

    Arquitectura (Navon et al. 2021, LeNetHyper):
      ray_mlp:  2 → ray_hidden → ray_hidden   (compartido)
      heads:    ray_hidden → tamaño de cada tensor de pesos
    """

    def __init__(self, ray_hidden: int = 100, gnn_hidden: int = 64, in_channels: int = 3):
        super().__init__()
        self.gnn_hidden  = gnn_hidden
        self.in_channels = in_channels
        h, c = gnn_hidden, in_channels

        # Base compartida
        self.ray_mlp = nn.Sequential(
            nn.Linear(2, ray_hidden), nn.LeakyReLU(0.1),
            nn.Linear(ray_hidden, ray_hidden), nn.LeakyReLU(0.1),
        )

        # Heads de generación de pesos
        # SAGEConv1 (in_channels → gnn_hidden)
        self.c1_ll_w = nn.Linear(ray_hidden, h * c)  # lin_l weight
        self.c1_ll_b = nn.Linear(ray_hidden, h)       # lin_l bias
        self.c1_lr_w = nn.Linear(ray_hidden, h * c)  # lin_r weight (sin bias)

        # SAGEConv2 (gnn_hidden → gnn_hidden)
        self.c2_ll_w = nn.Linear(ray_hidden, h * h)  # lin_l weight
        self.c2_ll_b = nn.Linear(ray_hidden, h)       # lin_l bias
        self.c2_lr_w = nn.Linear(ray_hidden, h * h)  # lin_r weight (sin bias)

        # Cabeza de clasificación (gnn_hidden → 1)
        self.fin_w = nn.Linear(ray_hidden, h)
        self.fin_b = nn.Linear(ray_hidden, 1)

    def forward(self, r: torch.Tensor) -> dict:
        """
        r: (B, 2)  →  dict con tensores (B, out_dim, in_dim) listos para F.linear
        """
        feat = self.ray_mlp(r)   # (B, ray_hidden)
        h, c = self.gnn_hidden, self.in_channels
        B = r.size(0)

        return {
            'c1_ll_w': self.c1_ll_w(feat).view(B, h, c),
            'c1_ll_b': self.c1_ll_b(feat),               # (B, h)
            'c1_lr_w': self.c1_lr_w(feat).view(B, h, c),
            'c2_ll_w': self.c2_ll_w(feat).view(B, h, h),
            'c2_ll_b': self.c2_ll_b(feat),               # (B, h)
            'c2_lr_w': self.c2_lr_w(feat).view(B, h, h),
            'fin_w':   self.fin_w(feat).view(B, 1, h),
            'fin_b':   self.fin_b(feat).view(B, 1),
        }

## 3. PHNGraphSAGE

GNN de 2 capas SAGEConv con BatchNorm y Dropout.  
La diferencia con el `SAGE` de Graph-SCP es que los pesos de `lin_l` / `lin_r` se pasan
como argumento en lugar de ser parámetros del módulo — los genera el `HyperNetwork`.

Los únicos parámetros propios del GNN son las dos capas BatchNorm (compartidas para
todos los r; normalizan las activaciones independientemente de la preferencia).

In [4]:
class PHNGraphSAGE(nn.Module):
    """
    2-layer GraphSAGE con pesos externos.

    Adaptado de Graph-SCP (Shafi et al. 2025), clase SAGE (utils.py L663).
    Capas propias: BatchNorm1d × 2  (compartidas, ~256 params).
    """

    def __init__(self, hidden: int = 64, dropout: float = 0.4):
        super().__init__()
        self.aggregator = MeanAggregator()
        self.bn1        = nn.BatchNorm1d(hidden)
        self.bn2        = nn.BatchNorm1d(hidden)
        self.dropout    = dropout

    def _sage_layer(self, x, edge_index, ll_w, ll_b, lr_w, N):
        """
        Una capa SAGEConv con pesos externos.
          agg[i] = mean de features de vecinos de i
          out[i] = ll_w @ agg[i] + ll_b  +  lr_w @ x[i]
        """
        agg = self.aggregator(x, edge_index, N)           # (N, in)
        return F.linear(agg, ll_w, ll_b) + F.linear(x, lr_w)  # (N, out)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, w: dict) -> torch.Tensor:
        """
        x          : (N, in_channels)  features de nodos de UN grafo
        edge_index : (2, E)            índices locales [0, N)
        w          : dict con los pesos del HyperNetwork para este grafo
        returns    : (N,) probabilidades en [0,1]
        """
        N = x.size(0)

        # Capa 1: (N, in_channels) → (N, hidden)
        h = self._sage_layer(x, edge_index, w['c1_ll_w'], w['c1_ll_b'], w['c1_lr_w'], N)
        h = self.bn1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        # Capa 2: (N, hidden) → (N, hidden)
        h = self._sage_layer(h, edge_index, w['c2_ll_w'], w['c2_ll_b'], w['c2_lr_w'], N)
        h = self.bn2(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        # Clasificación: (N, hidden) → (N, 1) → (N,)
        out = F.linear(h, w['fin_w'], w['fin_b'])  # fin_w: (1, h), fin_b: (1,)
        return torch.sigmoid(out).squeeze(-1)

## 4. PHNModel — modelo completo

Combina HPN + GNN. Recibe un batch de PyG y devuelve las probabilidades por nodo,
procesando cada grafo del batch de forma independiente con sus pesos específicos.

In [5]:
class PHNModel(nn.Module):
    """
    Pareto HyperNetwork + GraphSAGE para colocación óptima de cámaras.

    Parámetros
    ----------
    in_channels : features de entrada por nodo (default 3: [flag, prob, degree])
    gnn_hidden  : dimensión oculta del GraphSAGE (default 64)
    ray_hidden  : dimensión oculta del ray_mlp del HPN (default 100)
    dropout     : tasa de dropout en el GNN (default 0.4)
    """

    def __init__(
        self,
        in_channels: int = 3,
        gnn_hidden:  int = 64,
        ray_hidden:  int = 100,
        dropout:   float = 0.4,
    ):
        super().__init__()
        self.hnet = HyperNetwork(ray_hidden, gnn_hidden, in_channels)
        self.gnn  = PHNGraphSAGE(gnn_hidden, dropout)

    def forward(self, batch) -> torch.Tensor:
        """
        batch : PyG Batch — varios grafos concatenados.
                Debe tener atributos: x, edge_index, batch, r
        returns: (N_total,) probabilidades en [0,1] en el mismo orden que batch.x
        """
        r = batch.r.view(batch.num_graphs, 2)  # (B, 2)
        b = batch.batch                          # (N_total,)

        # Generar pesos para todos los grafos del batch en paralelo
        weights = self.hnet(r)   # dict: cada valor tiene shape (B, ...)

        outputs     = []
        node_offset = 0

        for i in range(batch.num_graphs):
            mask_i = (b == i)
            x_i    = batch.x[mask_i]           # (N_i, in_channels)
            N_i    = x_i.size(0)

            # Aristas de este grafo → reindexar a [0, N_i)
            e_mask = mask_i[batch.edge_index[0]]        # fuentes que pertenecen al grafo i
            ei_i   = batch.edge_index[:, e_mask] - node_offset

            # Pesos para el grafo i: (B, ...) → (...)
            w_i = {k: v[i] for k, v in weights.items()}

            out_i = self.gnn(x_i, ei_i, w_i)
            outputs.append(out_i)
            node_offset += N_i

        return torch.cat(outputs, dim=0)  # (N_total,)

    def count_parameters(self):
        hnet_p = sum(p.numel() for p in self.hnet.parameters())
        gnn_p  = sum(p.numel() for p in self.gnn.parameters())
        print("=" * 35)
        print(f"  HyperNetwork      : {hnet_p:>10,}")
        print(f"  GNN (BatchNorm)   : {gnn_p:>10,}")
        print(f"  Total             : {hnet_p+gnn_p:>10,}")
        print("=" * 35)
        return hnet_p + gnn_p

## 5. Instanciar y contar parámetros

In [6]:
model = PHNModel(
    in_channels = 3,     # [flag, prob, degree]
    gnn_hidden  = 64,
    ray_hidden  = 100,
    dropout     = 0.4,
).to(DEVICE)

model.count_parameters()
print()
print(model)

  HyperNetwork      :    896,069
  GNN (BatchNorm)   :        256
  Total             :    896,325

PHNModel(
  (hnet): HyperNetwork(
    (ray_mlp): Sequential(
      (0): Linear(in_features=2, out_features=100, bias=True)
      (1): LeakyReLU(negative_slope=0.1)
      (2): Linear(in_features=100, out_features=100, bias=True)
      (3): LeakyReLU(negative_slope=0.1)
    )
    (c1_ll_w): Linear(in_features=100, out_features=192, bias=True)
    (c1_ll_b): Linear(in_features=100, out_features=64, bias=True)
    (c1_lr_w): Linear(in_features=100, out_features=192, bias=True)
    (c2_ll_w): Linear(in_features=100, out_features=4096, bias=True)
    (c2_ll_b): Linear(in_features=100, out_features=64, bias=True)
    (c2_lr_w): Linear(in_features=100, out_features=4096, bias=True)
    (fin_w): Linear(in_features=100, out_features=64, bias=True)
    (fin_b): Linear(in_features=100, out_features=1, bias=True)
  )
  (gnn): PHNGraphSAGE(
    (aggregator): MeanAggregator()
    (bn1): BatchNorm1d(64,

## 6. Verificar forward pass con un batch real

Cargamos el dataset de entrenamiento (del notebook 04) y pasamos un mini-batch.

In [7]:
import json
import numpy as np
from torch.utils.data import Dataset
from torch_geometric.data import Data

GRAPH_DIR  = ROOT / "graphs"
PARETO_DIR = ROOT / "frentes_pareto_resultados" / "clean"

with open(GRAPH_DIR / "split.json") as f:
    split = json.load(f)

# ── Copiar ParetoDataset del notebook 04 ──────────────────────────────────────
class ParetoDataset(Dataset):
    def __init__(self, instance_names, graph_dir, pareto_dir, n_pref=1, seed=None):
        super().__init__()
        self.n_pref = n_pref
        self.rng    = np.random.default_rng(seed)
        self.instances = []
        for name in instance_names:
            g_path = Path(graph_dir) / (name + ".pt")
            o_path = Path(pareto_dir) / name / "objectives.npy"
            s_path = Path(pareto_dir) / name / "solutions.npy"
            if not g_path.exists() or not o_path.exists():
                continue
            graph = torch.load(g_path, weights_only=False)
            self.instances.append({
                "name": name,
                "graph": graph,
                "objectives": np.load(o_path),
                "solutions":  np.load(s_path),
            })
        print(f"Cargadas {len(self.instances)} instancias")

    def __len__(self):
        return len(self.instances) * self.n_pref

    def __getitem__(self, idx):
        inst  = self.instances[idx % len(self.instances)]
        graph = inst["graph"]
        r     = self._sample_r()
        y_star = self._dynamic_match(inst["objectives"], inst["solutions"], r, graph)
        data = Data(x=graph.x, edge_index=graph.edge_index)
        data.instance_name = graph.instance_name
        data.n_candidates  = graph.n_candidates
        data.cov_max       = graph.cov_max
        data.n_pareto      = graph.n_pareto
        data.r      = torch.tensor(r,      dtype=torch.float)
        data.y_star = torch.tensor(y_star, dtype=torch.float)
        return data

    def _sample_r(self):
        r1 = self.rng.uniform(0.0, 1.0)
        return np.array([r1, 1.0 - r1], dtype=np.float32)

    def _dynamic_match(self, objectives, solutions, r, graph):
        coverage  = -objectives[:, 0]
        cost      =  objectives[:, 1]
        cov_norm  = coverage / float(graph.cov_max)
        cost_norm = cost     / float(graph.n_candidates)
        scores    = r[0] * (1.0 - cov_norm) + r[1] * cost_norm
        return solutions[int(np.argmin(scores))].astype(np.float32)


ds_train = ParetoDataset(split["train"], GRAPH_DIR, PARETO_DIR, n_pref=1, seed=7)
loader   = DataLoader(ds_train, batch_size=4, shuffle=True, num_workers=0)

Cargadas 800 instancias


In [8]:
batch = next(iter(loader))
batch = batch.to(DEVICE)

model.eval()
with torch.no_grad():
    probs = model(batch)

print(f"batch.x          : {tuple(batch.x.shape)}")
print(f"batch.edge_index : {tuple(batch.edge_index.shape)}")
print(f"batch.r (raw)    : {tuple(batch.r.shape)}  → r per graph: {batch.r.view(batch.num_graphs,2).tolist()}")
print(f"batch.y_star     : {tuple(batch.y_star.shape)}")
print()
print(f"probs            : shape={tuple(probs.shape)}")
print(f"  min={probs.min():.4f}  max={probs.max():.4f}  mean={probs.mean():.4f}")
print(f"  Todos en [0,1]: {((probs >= 0) & (probs <= 1)).all().item()}")

batch.x          : (600, 3)
batch.edge_index : (2, 4320)
batch.r (raw)    : (8,)  → r per graph: [[0.6250954866409302, 0.3749045431613922], [0.8972138166427612, 0.10278619825839996], [0.7756856679916382, 0.22431431710720062], [0.22520719468593597, 0.7747927904129028]]
batch.y_star     : (600,)

probs            : shape=(600,)
  min=0.2155  max=0.5317  mean=0.4273
  Todos en [0,1]: True


## 7. Verificar gradientes (backward pass)

In [9]:
model.train()
batch = next(iter(loader)).to(DEVICE)

probs  = model(batch)
y_star = batch.y_star

# Binary Cross-Entropy con logits se usa usualmente, pero como ya tenemos sigmoid
# usamos BCE directo. Añadimos clip para estabilidad numérica.
loss = F.binary_cross_entropy(probs.clamp(1e-6, 1 - 1e-6), y_star)
loss.backward()

print(f"Loss: {loss.item():.4f}")

# Verificar que todos los parámetros del HPN recibieron gradiente
no_grad = [n for n, p in model.named_parameters() if p.grad is None]
with_grad = [n for n, p in model.named_parameters() if p.grad is not None]
print(f"\nParámetros con gradiente : {len(with_grad)}")
print(f"Parámetros sin gradiente : {len(no_grad)}")
if no_grad:
    print(f"  Sin grad: {no_grad}")

Loss: 0.6819

Parámetros con gradiente : 24
Parámetros sin gradiente : 0


## 8. Efecto de r en los pesos generados

Verificamos que el HPN genera pesos distintos para distintas preferencias.

In [10]:
model.eval()
r_vals = [
    (1.0, 0.0),  # solo cobertura
    (0.5, 0.5),  # equilibrado
    (0.0, 1.0),  # solo costo
]

print("Norma de los pesos generados por el HPN según r:")
print(f"{'r_cov':>6} {'r_cost':>6}  {'c1_ll_w':>10}  {'c2_ll_w':>10}  {'fin_w':>8}")

with torch.no_grad():
    for r_cov, r_cost in r_vals:
        r_t = torch.tensor([[r_cov, r_cost]], dtype=torch.float, device=DEVICE)
        w   = model.hnet(r_t)
        print(
            f"  {r_cov:.1f}    {r_cost:.1f}  "
            f"{w['c1_ll_w'].norm().item():10.4f}  "
            f"{w['c2_ll_w'].norm().item():10.4f}  "
            f"{w['fin_w'].norm().item():8.4f}"
        )

# Diferencia entre pesos extremos
r_a = torch.tensor([[1.0, 0.0]], device=DEVICE)
r_b = torch.tensor([[0.0, 1.0]], device=DEVICE)
wa  = model.hnet(r_a)
wb  = model.hnet(r_b)
diff = (wa['c2_ll_w'] - wb['c2_ll_w']).norm().item()
print(f"\nDiferencia ‖W_c2(r=cov) − W_c2(r=cost)‖ = {diff:.4f}  (debe ser > 0)")

Norma de los pesos generados por el HPN según r:
 r_cov r_cost     c1_ll_w     c2_ll_w     fin_w
  1.0    0.0      1.6289      7.4365    0.7818
  0.5    0.5      1.5833      7.1938    0.7813
  0.0    1.0      1.7567      8.1004    0.9432

Diferencia ‖W_c2(r=cov) − W_c2(r=cost)‖ = 5.6258  (debe ser > 0)


## Resumen de la arquitectura

```
PHNModel
├── HyperNetwork
│   ├── ray_mlp:  Linear(2→100) → ReLU → Linear(100→100) → ReLU
│   ├── c1_ll_w:  Linear(100 → 64×3 = 192)     → reshape (64, 3)
│   ├── c1_ll_b:  Linear(100 → 64)
│   ├── c1_lr_w:  Linear(100 → 192)             → reshape (64, 3)
│   ├── c2_ll_w:  Linear(100 → 64×64 = 4096)   → reshape (64, 64)
│   ├── c2_ll_b:  Linear(100 → 64)
│   ├── c2_lr_w:  Linear(100 → 4096)            → reshape (64, 64)
│   ├── fin_w:    Linear(100 → 64)               → reshape (1, 64)
│   └── fin_b:    Linear(100 → 1)
│
└── PHNGraphSAGE
    ├── MeanAggregator  (sin parámetros propios)
    ├── BatchNorm1d(64)  ← únicos params compartidos del GNN
    └── BatchNorm1d(64)
```